<a href="https://colab.research.google.com/github/adyogyy-creator/ady.ogy.ipynb/blob/main/V1_Circuit_Parser_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# V2 - Extract numerical features from the circuit graph

def extract_graph_features(G):
    features = {
        "num_nodes": G.number_of_nodes(),
        "num_edges": G.number_of_edges(),
        "num_inputs": sum(
            1 for _, data in G.nodes(data=True)
            if data.get("type") == "INPUT"
        ),
        "num_outputs": sum(
            1 for _, data in G.nodes(data=True)
            if data.get("type") == "OUTPUT"
        ),
        "num_gates": sum(
            1 for _, data in G.nodes(data=True)
            if data.get("type") not in ["INPUT", "OUTPUT"]
        ),
    }

    return features


features = extract_graph_features(G)

print("Circuit features:")
for name, value in features.items():
    print(f"{name}: {value}")

Circuit features:
num_nodes: 7
num_edges: 6
num_inputs: 4
num_outputs: 1
num_gates: 2


In [12]:
# V2 - Create a small synthetic circuit dataset

circuits = {
    "circuit_1": """
    AND gate1 a b n1
    OR gate2 n1 c out
    """,

    "circuit_2": """
    AND gate1 a b n1
    AND gate2 c d n2
    OR gate3 n1 n2 out
    """,

    "circuit_3": """
    OR gate1 a b n1
    AND gate2 n1 c n2
    NOT gate3 n2 out
    """,

    "circuit_4": """
    AND gate1 a b n1
    OR gate2 c d n2
    AND gate3 n1 n2 n3
    OR gate4 n3 e out
    """
}

dataset = []

for circuit_name, netlist in circuits.items():

    parsed_circuit = parse_netlist(netlist)

    graph = nx.DiGraph()

    for gate in parsed_circuit:
        graph.add_node(
            gate["name"],
            type=gate["type"]
        )

        for signal in gate["inputs"]:
            graph.add_node(
                signal,
                type="INPUT"
            )
            graph.add_edge(signal, gate["name"])

        output = gate["output"]

        graph.add_node(
            output,
            type="OUTPUT"
        )

        graph.add_edge(gate["name"], output)

    features = extract_graph_features(graph)

    features["circuit"] = circuit_name

    dataset.append(features)

for row in dataset:
    print(row)

{'num_nodes': 7, 'num_edges': 6, 'num_inputs': 4, 'num_outputs': 1, 'num_gates': 2, 'circuit': 'circuit_1'}
{'num_nodes': 10, 'num_edges': 9, 'num_inputs': 6, 'num_outputs': 1, 'num_gates': 3, 'circuit': 'circuit_2'}
{'num_nodes': 9, 'num_edges': 8, 'num_inputs': 5, 'num_outputs': 1, 'num_gates': 3, 'circuit': 'circuit_3'}
{'num_nodes': 13, 'num_edges': 12, 'num_inputs': 8, 'num_outputs': 1, 'num_gates': 4, 'circuit': 'circuit_4'}


In [13]:
# V2 - Create ML features and a synthetic target

import pandas as pd

df = pd.DataFrame(dataset)

# Synthetic target for this prototype.
# This is deliberately simple so we can validate the ML pipeline.
df["complexity_score"] = (
    df["num_gates"] * 2
    + df["num_edges"] * 0.5
    + df["num_inputs"] * 0.25
)

print(df)

   num_nodes  num_edges  num_inputs  num_outputs  num_gates    circuit  \
0          7          6           4            1          2  circuit_1   
1         10          9           6            1          3  circuit_2   
2          9          8           5            1          3  circuit_3   
3         13         12           8            1          4  circuit_4   

   complexity_score  
0              8.00  
1             12.00  
2             11.25  
3             16.00  


In [14]:
# Separate input features from the target

feature_columns = [
    "num_nodes",
    "num_edges",
    "num_inputs",
    "num_outputs",
    "num_gates"
]

X = df[feature_columns]
y = df["complexity_score"]

print("Features:")
print(X)

print("\nTargets:")
print(y)

Features:
   num_nodes  num_edges  num_inputs  num_outputs  num_gates
0          7          6           4            1          2
1         10          9           6            1          3
2          9          8           5            1          3
3         13         12           8            1          4

Targets:
0     8.00
1    12.00
2    11.25
3    16.00
Name: complexity_score, dtype: float64


In [15]:
# V2 - Train a baseline machine-learning model

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

model = LinearRegression()

model.fit(X, y)

predictions = model.predict(X)

mae = mean_absolute_error(y, predictions)

print("Predictions:")
for actual, predicted in zip(y, predictions):
    print(f"Actual: {actual:.2f} | Predicted: {predicted:.2f}")

print(f"\nMean Absolute Error: {mae:.4f}")

Predictions:
Actual: 8.00 | Predicted: 8.00
Actual: 12.00 | Predicted: 12.00
Actual: 11.25 | Predicted: 11.25
Actual: 16.00 | Predicted: 16.00

Mean Absolute Error: 0.0000


In [17]:
# V2 - Train/test evaluation

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Test predictions:")

for actual, predicted in zip(y_test, y_pred):
    print(f"Actual: {actual:.2f} | Predicted: {predicted:.2f}")

print(f"\nTest MAE: {mae:.4f}")
print(f"Test R²: {r2:.4f}")

Test predictions:
Actual: 12.00 | Predicted: 12.00

Test MAE: 0.0000
Test R²: nan


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


In [18]:
# V2.1 - Generate synthetic circuit dataset

import random
import pandas as pd

random.seed(42)

gate_types = ["AND", "OR", "NOT"]

def generate_random_circuit(circuit_id, min_gates=2, max_gates=12):
    num_gates = random.randint(min_gates, max_gates)

    inputs = ["in0", "in1", "in2", "in3", "in4", "in5"]
    signals = inputs.copy()

    netlist = []

    for i in range(num_gates):
        gate_type = random.choice(gate_types)
        gate_name = f"gate_{i}"

        if gate_type == "NOT":
            input_signal = random.choice(signals)
            output_signal = f"n{i}"

            netlist.append(
                f"NOT {gate_name} {input_signal} {output_signal}"
            )

        else:
            input_a = random.choice(signals)
            input_b = random.choice(signals)
            output_signal = f"n{i}"

            netlist.append(
                f"{gate_type} {gate_name} {input_a} {input_b} {output_signal}"
            )

        signals.append(output_signal)

    final_signal = random.choice(signals)

    netlist.append(
        f"BUF output_gate {final_signal} out"
    )

    return "\n".join(netlist)


def build_graph_from_netlist(netlist):
    parsed_circuit = parse_netlist(netlist)

    graph = nx.DiGraph()

    for gate in parsed_circuit:
        graph.add_node(
            gate["name"],
            type=gate["type"]
        )

        for signal in gate["inputs"]:
            if signal not in graph:
                graph.add_node(
                    signal,
                    type="SIGNAL"
                )

            graph.add_edge(signal, gate["name"])

        output = gate["output"]

        graph.add_node(
            output,
            type="SIGNAL"
        )

        graph.add_edge(gate["name"], output)

    return graph


rows = []

for i in range(500):

    netlist = generate_random_circuit(i)

    graph = build_graph_from_netlist(netlist)

    features = extract_graph_features(graph)

    # Synthetic target for this prototype.
    complexity_score = (
        features["num_gates"] * 2
        + features["num_edges"] * 0.5
        + features["num_inputs"] * 0.25
    )

    features["circuit_id"] = i
    features["complexity_score"] = complexity_score

    rows.append(features)


large_df = pd.DataFrame(rows)

print("Dataset shape:", large_df.shape)
print()
print(large_df.head())

Dataset shape: (500, 7)

   num_nodes  num_edges  num_inputs  num_outputs  num_gates  circuit_id  \
0         30         30           0            0         30           0   
1         14         13           0            0         14           1   
2         21         19           0            0         21           2   
3         31         33           0            0         31           3   
4          9          8           0            0          9           4   

   complexity_score  
0              75.0  
1              34.5  
2              51.5  
3              78.5  
4              22.0  


In [19]:
# V2.1 - Train and evaluate on 500 synthetic circuits

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

feature_columns = [
    "num_nodes",
    "num_edges",
    "num_inputs",
    "num_outputs",
    "num_gates"
]

X_large = large_df[feature_columns]
y_large = large_df["complexity_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X_large,
    y_large,
    test_size=0.20,
    random_state=42
)

model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print()
print(f"Test MAE: {mae:.4f}")
print(f"Test R²: {r2:.4f}")

Training samples: 400
Test samples: 100

Test MAE: 0.0000
Test R²: 1.0000
